In [1]:
import sqlite3
import pandas as pd
import os

os.makedirs('data/powerbi', exist_ok=True)

print("=" * 80)
print("EXPORTING QUERY RESULTS TO CSV FOR POWER BI")
print("=" * 80)

# Connect to SQLite database
conn = sqlite3.connect('humanitarian_health.db')

# Query 1: All countries
print("\n[1/8] Exporting countries...")
df_q1 = pd.read_sql_query("SELECT * FROM countries;", conn)
df_q1.to_csv('data/powerbi/q1_countries.csv', index=False)
print(f"   Saved: {len(df_q1)} rows")

# Query 3: Highest mortality 2022
print("\n[2/8] Exporting highest mortality countries...")
df_q3 = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.region,
    h.year,
    h.under_5_mortality_rate
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
WHERE h.year = 2022
ORDER BY h.under_5_mortality_rate DESC
LIMIT 15;
""", conn)
df_q3.to_csv('data/powerbi/q3_highest_mortality.csv', index=False)
print(f"   Saved: {len(df_q3)} rows")

# Query 4: Regional averages
print("\n[3/8] Exporting regional averages...")
df_q4 = pd.read_sql_query("""
SELECT 
    c.region,
    COUNT(DISTINCT c.country_id) as num_countries,
    ROUND(AVG(h.under_5_mortality_rate), 2) as avg_mortality,
    ROUND(MAX(h.under_5_mortality_rate), 2) as highest_mortality,
    ROUND(MIN(h.under_5_mortality_rate), 2) as lowest_mortality,
    ROUND(AVG(h.life_expectancy), 1) as avg_life_expectancy
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
WHERE h.year = 2022
GROUP BY c.region
ORDER BY avg_mortality DESC;
""", conn)
df_q4.to_csv('data/powerbi/q4_regional_averages.csv', index=False)
print(f"   Saved: {len(df_q4)} rows")

# Query 5: Kenya trends (sample country)
print("\n[4/8] Exporting country trends...")
df_q5 = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.region,
    h.year,
    h.under_5_mortality_rate,
    h.life_expectancy,
    h.tuberculosis_incidence
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
ORDER BY c.country_name, h.year;
""", conn)
df_q5.to_csv('data/powerbi/q5_all_country_trends.csv', index=False)
print(f"   Saved: {len(df_q5)} rows")

# Query 6: Improvement vs worsening
print("\n[5/8] Exporting improvement analysis...")
df_q6 = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.region,
    h_2010.under_5_mortality_rate as mortality_2010,
    h_2022.under_5_mortality_rate as mortality_2022,
    ROUND(h_2022.under_5_mortality_rate - h_2010.under_5_mortality_rate, 2) as change,
    ROUND(((h_2022.under_5_mortality_rate - h_2010.under_5_mortality_rate) / h_2010.under_5_mortality_rate * 100), 1) as pct_change
FROM health_indicators h_2010
JOIN health_indicators h_2022 ON h_2010.country_id = h_2022.country_id
JOIN countries c ON h_2010.country_id = c.country_id
WHERE h_2010.year = 2010 AND h_2022.year = 2022
ORDER BY change DESC;
""", conn)
df_q6.to_csv('data/powerbi/q6_improvement_analysis.csv', index=False)
print(f"   Saved: {len(df_q6)} rows")

# Query 7: Regional rankings
print("\n[6/8] Exporting regional rankings...")
df_q7 = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.region,
    h.year,
    h.under_5_mortality_rate,
    RANK() OVER (PARTITION BY c.region, h.year ORDER BY h.under_5_mortality_rate DESC) as rank_in_region
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
WHERE h.year = 2022
ORDER BY c.region, rank_in_region;
""", conn)
df_q7.to_csv('data/powerbi/q7_regional_rankings.csv', index=False)
print(f"   Saved: {len(df_q7)} rows")

# Query 8: Year-over-year changes
print("\n[7/8] Exporting year-over-year changes...")
df_q8 = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.region,
    h.year,
    h.under_5_mortality_rate,
    LAG(h.under_5_mortality_rate) OVER (PARTITION BY h.country_id ORDER BY h.year) as prev_year_mortality,
    ROUND(h.under_5_mortality_rate - LAG(h.under_5_mortality_rate) OVER (PARTITION BY h.country_id ORDER BY h.year), 2) as year_over_year_change
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
ORDER BY c.country_name, h.year;
""", conn)
df_q8.to_csv('data/powerbi/q8_year_over_year.csv', index=False)
print(f"   Saved: {len(df_q8)} rows")

# Master table for Power BI (all 2022 data)
print("\n[8/8] Exporting master data table...")
df_master = pd.read_sql_query("""
SELECT 
    c.country_name,
    c.country_code,
    c.region,
    h.year,
    h.under_5_mortality_rate,
    h.life_expectancy,
    h.tuberculosis_incidence,
    CASE 
        WHEN h.under_5_mortality_rate > 8 THEN 'Critical'
        WHEN h.under_5_mortality_rate > 6 THEN 'High'
        ELSE 'Moderate'
    END as health_status
FROM health_indicators h
JOIN countries c ON h.country_id = c.country_id
ORDER BY c.region, c.country_name, h.year;
""", conn)
df_master.to_csv('data/powerbi/master_health_data.csv', index=False)
print(f"   Saved: {len(df_master)} rows")

conn.close()

print("\n" + "=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)
print("\nCSV files created in: data/powerbi/")
print("\nFiles ready for Power BI:")
print("  ✓ q1_countries.csv")
print("  ✓ q3_highest_mortality.csv")
print("  ✓ q4_regional_averages.csv")
print("  ✓ q5_all_country_trends.csv")
print("  ✓ q6_improvement_analysis.csv")
print("  ✓ q7_regional_rankings.csv")
print("  ✓ q8_year_over_year.csv")
print("  ✓ master_health_data.csv")
print("\nReady to import into Power BI!")

EXPORTING QUERY RESULTS TO CSV FOR POWER BI

[1/8] Exporting countries...
   Saved: 15 rows

[2/8] Exporting highest mortality countries...
   Saved: 15 rows

[3/8] Exporting regional averages...
   Saved: 8 rows

[4/8] Exporting country trends...
   Saved: 195 rows

[5/8] Exporting improvement analysis...
   Saved: 15 rows

[6/8] Exporting regional rankings...
   Saved: 15 rows

[7/8] Exporting year-over-year changes...
   Saved: 195 rows

[8/8] Exporting master data table...
   Saved: 195 rows

EXPORT COMPLETE

CSV files created in: data/powerbi/

Files ready for Power BI:
  ✓ q1_countries.csv
  ✓ q3_highest_mortality.csv
  ✓ q4_regional_averages.csv
  ✓ q5_all_country_trends.csv
  ✓ q6_improvement_analysis.csv
  ✓ q7_regional_rankings.csv
  ✓ q8_year_over_year.csv
  ✓ master_health_data.csv

Ready to import into Power BI!
